# Guide to getting started authoring your own protocol registry YAML

For: The SCOOP project

Last updated: 2026/07/10 by Brianna Argall

First, take a rough look at an existing profile, such as nsclc, to get an idea of what the final YAML can look like. 

When you are writing the YAML, it is highly recommended that you set up a notebook and that you use this while you are writing your YAML protocol registry. Give the notebook a descriptive name and write notes for yourself in markdown cells to help you remember the purpose of your notebook.

## Get required imports working:

In the example notebooks to help you, the first block of code is something like:

In [1]:
from pathlib import Path
import os
import sys

# Here we define the function that lets SCOOP be found from within the SCOOP
# folder.  This allows navigation to files inside SCOOP to work.
# It defines the repo (code repository) root as containing a notebooks and scoop folder
# If you not working inside the SCOOP repository it will not work - which is
# fine as long as you can import SCOOP - only use this function if relevant to you.
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repo root.")

# Here we call on the function to make sure our code can find SCOOP
ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_ROOT = ROOT / "scoop" / "configuration"

Whether you are working in the notebooks folder inside SCOOP or not, make sure you can import from scoop / a file within SCOOP. 

In [ ]:
from scoop.utils.treatment_selection import ConditionDevTools, show_conditions

In order to investigate what's available in the HemOncKB you will need access to the HemOncKB. This project expects access to data models from hemonc_import to understand the data requirements and relationships in the HemOncKB. hemonc_import is from the Australian Cancer Data Network (not HemOnc). If you do not have this and need it for some code to run you will get an error message telling you to install it.

For interactions with the database, you will need to pass a session to the relevant function. To do this, you can specify in your environment (using the .env file) the database engine url.

In [3]:
import sqlalchemy as sa
import sqlalchemy.orm as so
HEMONC_ENGINE_URL = os.environ["HEMONC_ENGINE_URL"]

def _require_engine_url(engine_url: str | None = None) -> str:
    resolved = engine_url or HEMONC_ENGINE_URL
    if not resolved:
        raise ValueError("Set HEMONC_ENGINE_URL or pass engine_url explicitly.")
    return resolved

engine = sa.create_engine(_require_engine_url(HEMONC_ENGINE_URL))

You should now have the engine which you can use to get session access. Let's start by looking up what conditions are available in your copy of the HemOncKB

In [4]:
with so.Session(engine) as session:
    display(show_conditions("lung", session)) # try changing "lung" 

,condition,condition_cui
0,ALK-positive Non-small cell lung cancer,30824
1,BRAF-mutated Non-small cell lung cancer,30898
2,c-Met-positive Non-small cell lung cancer,154361
3,EGFR-mutated Non-small cell lung cancer,30910
4,HER2-mutated Non-small cell lung cancer,52958
5,KRAS-mutated Non-small cell lung cancer,54044
6,Lung cancer,46104
7,MET-mutated Non-small cell lung cancer,13094
8,Non-small cell lung cancer,642
9,Non-small cell lung cancer nonsquamous,25309


SCOOP accepts conditions lists as either lists of strings or as lists of CUIs (as integers).
Spelling must be exact. Behind the scenes, strings are converted to CUIs by looking them up in the database. Therefore, sometimes if you choose strings rather than CUIs session must be passed.

## Create a helper with ConditionDevTools

Let's make a helper to see what components / component classes are available. When you specify a component in SCOOP it will check across relevant columns and return a match in any of those columns as a hit for that component/row.

Let's check which columns are specified as relevant:

In [5]:
from scoop.utils.hemonc_helpers import COMPONENT_STR_COLS
print(COMPONENT_STR_COLS)

['canmed_major_class', 'canmed_minor_class', 'main_class', 'component']


We now want to check what components (and their classes) are available for the conditions we are interested in. Change the conditions list to the conditions you want to look at, using the show conditions tool from earlier to help you find these. To help us do this, we'll make a helper.

In [6]:
MY_CONDITIONS = ["Lung cancer", "Non-small cell lung cancer"]

with so.Session(engine) as session:
    my_helper = ConditionDevTools(conditions=MY_CONDITIONS, session=session)

Our helper will only return results relevant to the conditions list used when making it.
We can use it to search for relevant components like how we searched for conditions before:

In [7]:
with so.Session(engine) as session:
    my_helper.show_component_with_substring(session, "met") # Try other searches

['Anti-c-Met antibody-drug conjugate', 'Antimetabolite', 'Binimetinib', 'Dexamethasone', 'MET inhibitor', 'Methotrexate', 'Metoclopramide', 'Pemetrexed', 'Trametinib']


But we can also produce a dataframe of all of our relevant components which we can explore like we would any other pandas dataframe. This may take a bit the first time, but if you ask for the same dataframe the helper will remember the first version and give that to you quickly.

In [8]:
with so.Session(engine) as session:
    my_components = my_helper.components.get_dataframe(session)

In [9]:
# id here refers to sig id. This is created by hemonc_import, not provided with HemOncKB
# it can be helpful for you to look up sigs for more info but do not
# expect it to be consistent across different versions
my_components.head()

,id,regimen,variant_cui,component,component_cui,drug_inn,main_class,canmed_major_class,canmed_minor_class,study,phase
0,3428.0,Carboplatin and Paclitaxel (CP),130386.0,Carboplatin,88.0,carboplatin,Platinum agent,Alkylating agent,Platinum agent,A4021016|A8501001|ATTRACT-1|ECOG E4599|IPASS|K...,NaN
1,3429.0,Carboplatin and Paclitaxel (CP),130386.0,Paclitaxel,379.0,paclitaxel,Taxane,Microtubule inhibitor,Taxane,A4021016|A8501001|ATTRACT-1|ECOG E4599|IPASS|K...,NaN
2,9967.0,Dacomitinib monotherapy,131467.0,Dacomitinib,133.0,dacomitinib,EGFR inhibitor,Tyrosine kinase inhibitor,EGFR inhibitor,A7471028|ARCHER 1009|ARCHER 1050,NaN
3,12665.0,Erlotinib monotherapy,131961.0,Erlotinib,198.0,erlotinib,EGFR inhibitor,Tyrosine kinase inhibitor,EGFR inhibitor,A7471028|ADVIGO 1018|ARCHER 1009|ARTEMIS-CTONG...,NaN
4,754.0,Afatinib and Bevacizumab,129625.0,Afatinib,13.0,afatinib,EGFR inhibitor,Tyrosine kinase inhibitor,EGFR inhibitor,ABC Study,NaN


In [10]:
my_components.info()

<class 'pandas.DataFrame'>
Index: 951 entries, 0 to 951
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  951 non-null    float64
 1   regimen             951 non-null    str    
 2   variant_cui         896 non-null    float64
 3   component           951 non-null    str    
 4   component_cui       951 non-null    float64
 5   drug_inn            889 non-null    str    
 6   main_class          912 non-null    str    
 7   canmed_major_class  815 non-null    str    
 8   canmed_minor_class  780 non-null    str    
 9   study               951 non-null    str    
 10  phase               193 non-null    str    
dtypes: float64(3), str(8)
memory usage: 208.2 KB


## Check broad category exists for each component

For non-small cell lung cancer I made sure every component maps to a broad category. One day I hope to make this mapping automatic - for now it is manually specified in class maps. If you are looking at another condition, one of your recommended first tasks is to check if there are any components for which broad category is not specifed. This is an example workflow 

In [11]:
with so.Session(engine) as session:
    my_categorised = my_helper.components.get_dataframe(session, categorise=True)

First we check which broad categories are represented for the components relevant to you:

In [12]:
my_categorised["broad_category"].value_counts(dropna = False)

broad_category
chemotherapy     727
immunotherapy    135
supportive        57
radiotherapy      32
Name: count, dtype: int64

If you see an NA category above, you will want to make sure all of those uncategorised components get a category mapping.

Note that these classifications are per component and do not consider context. For example, leucovorin is always classified as "supportive" currently.

In [13]:
my_categorised[my_categorised["broad_category"].isna()]

,id,regimen,variant_cui,component,component_cui,drug_inn,main_class,canmed_major_class,canmed_minor_class,study,phase,broad_category


My recommendation is that you specify using the bigger class groups where you can.
The class maps live in hemonc_helpers, where you can easily edit them.

If you add a new broad category e.g. endocrine, you will also need to edit protocol_registry.py so we know how to treat your new category. Seach for BroadCategory and edit where appropriate.

For example: 

`BroadCategory = Literal["chemotherapy", "radiotherapy", "immunotherapy", "surgery", "supportive",]` 

would become

`BroadCategory = Literal["chemotherapy", "radiotherapy", "immunotherapy", "surgery", "supportive", "endocrine"]`

and 

```
DEFAULT_CATEGORY_RESTRICTIONS: Final[Dict[BroadCategory, CategoryRestriction]] = {
    "chemotherapy":  {"type": "exact", "amount": 0},
    "radiotherapy":  {"type": "exact", "amount": 0},
    "immunotherapy": {"type": "exact", "amount": 0},
    "surgery":       {"type": "exact", "amount": 0},
    # NOTE: "supportive" intentionally excluded
}
```

would become:

```
DEFAULT_CATEGORY_RESTRICTIONS: Final[Dict[BroadCategory, CategoryRestriction]] = {
    "chemotherapy":  {"type": "exact", "amount": 0},
    "radiotherapy":  {"type": "exact", "amount": 0},
    "immunotherapy": {"type": "exact", "amount": 0},
    "surgery":       {"type": "exact", "amount": 0},
    "endocrine":     {"type": "exact", "amount": 0},
    # NOTE: "supportive" intentionally excluded
}
```

so that unless someone asks for an endocrine-including regimen, we assume that the presence of endocrine components excludes that regimen.

After you have updated the class maps in hemonc_helpers.py, restart your notebook so that the new version is imported and rerun the checks for NA components to see your progress. 

## Creating a source phase specification

Now that you know how to:
- find conditions you can use
- find relevant components for your condition group
- categorise these components if needed

You are ready to start building source phases.